The CTO at "Car-ing is sharing", a car sales and rental company, hired you to help prototype a chatbot app that addresses diverse inquiries using LLMs. She proposed you piloting the following tasks:

Use a pre-trained LLM to classify the sentiment of the five car reviews in the car_reviews.csv dataset, and evaluate the classification accuracy and F1 score of predictions.
Store the model outputs in predicted_labels, then extract the labels and map them onto a list of {0,1} integer binary labels called predictions.
Store the calculated metrics in accuracy_result and f1_result.
The company is recently attracting customers from Spain. Extract and pass the first two sentences of the first review in the dataset to an English-to-Spanish translation LLM. Calculate the BLEU score to assess translation quality, using the content in reference_translations.txt as references.
Store the translated text generated by the LLM in translated_review.
Store the BLEU score metric result in bleu_score.
The 2nd review in the dataset emphasizes brand aspects. Load an extractive QA LLM such as "deepset/minilm-uncased-squad2" to formulate the question "What did he like about the brand?" and obtain an answer.
Use question and context for the two variables containing the LLM inputs: question and context.
Store the actual text answer in answer.
Summarize the last review in the dataset, into approximately 50-55 tokens long. Store it in the variable summarized_text.

In [1]:
import pandas as pd
from transformers import pipeline, MarianTokenizer, MarianMTModel
import evaluate
import torch

In [2]:
data = {
    "review": [
        "I absolutely love this car. The ride is smooth and comfortable. The fuel economy is also excellent.",
        "I really like this brand because it offers reliable cars at affordable prices. Their customer service is excellent too.",
        "The car looks beautiful, but I was disappointed with the interior quality. Some parts feel very cheap.",
        "This was a fantastic rental experience. The vehicle was clean, comfortable, and easy to drive.",
        "I had a terrible experience with this car. It broke down several times and the rental company was not helpful at all."
    ],
    
    "sentiment": [
        "positive",
        "positive",
        "negative",
        "positive",
        "negative"
    ]
}

df = pd.DataFrame(data)
df.to_csv("car_reviews.csv", index=False)

In [3]:
df.head()

,review,sentiment
0,I absolutely love this car. The ride is smooth...,positive
1,I really like this brand because it offers rel...,positive
2,"The car looks beautiful, but I was disappointe...",negative
3,This was a fantastic rental experience. The ve...,positive
4,I had a terrible experience with this car. It ...,negative


In [4]:
# Creating the model

classifier = pipeline(task="sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [5]:
reviews = df["review"].tolist()
actual_labels = df["sentiment"].tolist()

In [6]:
print(reviews)

['I absolutely love this car. The ride is smooth and comfortable. The fuel economy is also excellent.', 'I really like this brand because it offers reliable cars at affordable prices. Their customer service is excellent too.', 'The car looks beautiful, but I was disappointed with the interior quality. Some parts feel very cheap.', 'This was a fantastic rental experience. The vehicle was clean, comfortable, and easy to drive.', 'I had a terrible experience with this car. It broke down several times and the rental company was not helpful at all.']


In [7]:
predicted_label = classifier(reviews)
predicted_label

[{'label': 'POSITIVE', 'score': 0.9998832941055298},
 {'label': 'POSITIVE', 'score': 0.9997567534446716},
 {'label': 'NEGATIVE', 'score': 0.999620795249939},
 {'label': 'POSITIVE', 'score': 0.9998297691345215},
 {'label': 'NEGATIVE', 'score': 0.9997997879981995}]

In [8]:
predictions = [1 if item["label"] == "POSITIVE" else 0 for item in predicted_label]

In [9]:
predictions

[1, 1, 0, 1, 0]

In [10]:
accuracy = evaluate.load("accuracy")
f1 = evaluate.load("f1")

In [11]:
accuracy_result = accuracy.compute(predictions=predictions, references = [1 if label == "positive" else 0 for label in actual_labels])
f1_result = f1.compute(predictions=predictions, references = [1 if label == "positive" else 0 for label in actual_labels])

print(f"Accuracy: {accuracy_result}\nF1_Score: {f1_result}")

Accuracy: {'accuracy': 1.0}
F1_Score: {'f1': 1.0}


In [12]:
first_review = reviews[0]
print(first_review)

I absolutely love this car. The ride is smooth and comfortable. The fuel economy is also excellent.


In [13]:
first_review = str(first_review)
first_review

'I absolutely love this car. The ride is smooth and comfortable. The fuel economy is also excellent.'

In [14]:
sentences = first_review.split(". ")
sentences

['I absolutely love this car',
 'The ride is smooth and comfortable',
 'The fuel economy is also excellent.']

In [15]:
first_2_sentence = sentences[0] + ". " + sentences[1]

In [16]:
print(first_2_sentence)

I absolutely love this car. The ride is smooth and comfortable


In [17]:
# Translation llm

model_name = "Helsinki-NLP/opus-mt-en-es"

tokenizer = MarianTokenizer.from_pretrained(model_name)

model = MarianMTModel.from_pretrained(model_name)

C:\Users\HP-PC\anaconda3\envs\Machine_Learning\Lib\site-packages\transformers\models\marian\tokenization_marian.py:176: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

In [18]:
tokenized_sentence = tokenizer(first_2_sentence, return_tensors="pt", padding=True, truncation=True)

In [19]:
outputs = model.generate(**tokenized_sentence)

In [20]:
translated_review = tokenizer.decode(outputs[0], skip_special_tokens=True)

In [21]:
print(translated_review)

Me encanta este coche. El viaje es suave y cómodo


In [22]:
# Question answering bot, using the second review as context

from transformers import AutoTokenizer, AutoModelForQuestionAnswering

model_name = "deepset/minilm-uncased-squad2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: deepset/minilm-uncased-squad2
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
context = reviews[1]

question = "What does he like about this brand?"

# Tokenize the context and question
inputs = tokenizer(
    question,
    context,
    return_tensors="pt",
    padding=True,
    truncation=True
)

In [32]:
# Run the model

model.eval()

with torch.no_grad():
    output = model(**inputs)

# Get the most likely start and end postition
start_index = torch.argmax(output.start_logits, dim=1).item()
end_index = torch.argmax(output.end_logits, dim=1).item()

# Extracting answer
if end_index>=start_index:
    answer_tokens = inputs["input_ids"][0][start_index:end_index+1]
    answer = tokenizer.decode(answer_tokens, skip_special_tokens=True)
else:
    answer="Sorry I do not have the answer in my knowledge base"

print(answer)

it offers reliable cars at affordable prices
